### Introduction to NVIDIA cuOpt and MIP

NVIDIA cuOpt is a high-performance, GPU-accelerated combinatorial optimization engine. Unlike traditional solvers strictly focused on CPUs (such as Gurobi or CPLEX in their default modes), which expend significant computational time trying to mathematically prove the exact optimal bound through complex Branch and Bound routines, cuOpt's MIP (Mixed-Integer Programming) module focuses its horsepower on hardware-accelerated primal heuristics. It utilizes a powerful combination of evolutionary algorithms, parallel domain propagation, and the Primal-Dual Hybrid Gradient (PDLP) method. This architecture allows it to find extremely high-quality feasible solutions in fractions of a second, even for industrial-scale problems with millions of variables and constraints.

#### Machinery Scheduling (Combine Harvester Utilization)

In agriculture, the harvest window is a high-stakes, time-sensitive period. If mature crops spend too much time in the field, moisture levels rise, deteriorating quality, and sudden rainstorms can wipe out an entire season's yield. To mitigate this risk, large farms and agricultural cooperatives hire third-party contractors (machinery + operators) to secure rapid field clearance at the lowest possible cost.

1. The Core Objective

The goal is to meet a strict daily harvest volume target while minimizing outsourcing expenditures. Unlike the planting model (where we wanted to maximize revenue), the goal here is the Minimization (MINIMIZE) of the total daily machinery rental cost.

2. The Resource Profiles (Fleet Options)

The grower can hire two types of harvesting crews from the market, each presenting a distinct tradeoff between daily rental cost and harvesting throughput:

• Type A Crew (Standard Fleet): More economical, costing R$ 1,500 per day, but operates at a lower throughput, harvesting 80 tons of grain per day.

• Type B Crew (High-Performance Fleet): More expensive, costing R$ 2,200 per day, but works faster, harvesting 130 tons of grain per day.

3. The Operational Constraints (The Bottlenecks)

To make the optimization practical, the model introduces two conflicting operational walls:
1. Minimum Production Target: The combined fleet must harvest at least 1,200 tons of grain per day to fulfill predefined delivery contracts with processing plants.
2. Labor / Logistical Cap: Due to farm housing limitations or local operator availability, the infrastructure can accommodate a maximum of 15 total crews working simultaneously in the fields.


##### Libraries

In [1]:
from cuopt.linear_programming.problem import Problem, INTEGER, MINIMIZE
from cuopt.linear_programming.solver_settings import SolverSettings

##### Instantiating the Problem

This line initializes the core data structure in Python that acts as a bridge to NVIDIA’s native C++/CUDA engine. cuOpt allocates memory arrays to host variables, objective parameters, and constraint blocks, packaging them for high-speed GPU transfer.

In [2]:
prob_maquinas = Problem("Escalonamento_Colheita")


##### Declaring Variables

In [4]:
# 1. Defina explicitamente os limites (mínimo 0, máximo None/infinito)
equipe_A = prob_maquinas.addVariable(vtype=INTEGER, name="Equipes_Tipo_A")
equipe_B = prob_maquinas.addVariable(vtype=INTEGER, name="Equipes_Tipo_B")

• vtype=INTEGER: The solver flags these indices as discrete variables. The underlying branch-and-bound or primal-dual hybrid gradient (PDLP) heuristics are informed that fractional results (such as hiring 4.5 crews) are forbidden.

• Note on bounds: In the standard script, omitting lb=0 and ub might cause the engine to default to restrictive bounds. Explicitly specifying lb=0 and an upper bound safety limit (like ub=100) ensures the pre-solver does not lock the decision variables at zero during array scaling.

##### Objective Function

This line mathematically binds a cost coefficient vector (\(c = [1500, 2200]^T\)) to the decision variables. Crucially, the flag sense=MINIMIZE changes the direction of the optimization search. Instead of expanding outward for a ceiling, the engine shifts its focus to find the lowest possible intersection point inside the feasible region.

In [6]:
prob_maquinas.setObjective(1500 * equipe_A + 2200 * equipe_B, sense=MINIMIZE)

##### Constraints

• >= 1200: Creates a lower boundary wall. The fleet needs to harvest at least 1200 tons.Team A can harvest 80 tons and Team B can harvest 130 tons.

• <= 15: This is the maximum number of crew available

In [7]:
# 3. Adicione as restrições explicitamente
prob_maquinas.addConstraint(80 * equipe_A + 130 * equipe_B >= 1200, name="Meta_Producao")
prob_maquinas.addConstraint(equipe_A + equipe_B <= 15, name="Limite_Mao_Obra")


##### Solving

This acts as the processing trigger. Python yields control to the hardware. The Papilo presolver simplifies structural relationships on the CPU, and then hands the compressed matrix over to the CUDA Cores and Tensor Cores of your NVIDIA A10 to run massive genetic and primal-dual algorithms in milliseconds.

In [8]:
settings = SolverSettings()
#settings.time_limit = 10.0
solucao_maquinas = prob_maquinas.solve(SolverSettings())

cuOpt version: 26.8.0, git hash: 400863c1, host arch: x86_64, device archs: 70-real,75-real,80-real,86-real,90a-real,100f-real,120a-real,120
CPU: Intel(R) Xeon(R) Platinum 8358 CPU @ 2.60GHz, threads (physical/logical): 15/30, RAM (available/total): 228.80 / 235.93 GiB
CUDA 12.9, device: NVIDIA A10 (ID 0), VRAM: 22.06 GiB
CUDA device UUID: ca1b9d14-0fa2-5b1d-2f3e-e3df2112e63c
Solving a problem with 2 constraints, 2 variables (2 integers), and 4 nonzeros
Problem scaling:
Objective coefficents range:          [2e+03, 2e+03]
Constraint matrix coefficients range: [1e+00, 1e+02]
Constraint rhs / bounds range:        [0e+00, 1e+03]
Variable bounds range:                [0e+00, 0e+00]

MIP row scaling completed
New solution from early primal heuristics (CPUFJ). Objective +2.200000e+04. Time 0.02

Running Papilo presolve (git hash 32b3a87d)
Presolve status: reduced the problem
Presolve removed: 0 constraints, 0 variables, 0 nonzeros
Presolved problem: 2 constraints, 2 variables (2 integer), 4 


1. Hardware & Problem Ingestion

• Environment: The system successfully loaded cuOpt 26.8.0 on an NVIDIA A10 GPU utilizing 22.06 GiB of VRAM.

• The Model: It imported your specific problem consisting of 2 variables (Crews A and B) and 2 constraints (Production Target and Labor Limit).

2. Fast Heuristic Guesses (Primal Search)

• Before diving into deep mathematical proofs, cuOpt’s fast GPU engine (CPUFJ) immediately started guessing valid operational plans.

• It progressively lowered the daily operational cost across three quick iterations: from R$ 22,000 → R$ 21,500 → and finally down to R$ 20,600 in just 0.02 seconds per guess.

3. Continuous Baseline (LP Relaxation)

• To prove if R$ 20,600 was truly the cheapest option, the GPU temporarily ignored the "whole crew" integer rule and ran a fractional calculation (Dual Simplex).

• It found that if you could hire decimals of a crew, the absolute lowest physical baseline cost would be R$ 20,307.69 (Root relaxation objective).

4. Mathematical Cut & Final Verification

• The Slicing Plane: Since fractional crews are impossible, the solver applied a mathematical filter called a Strong CG Cut to slice away the fractional answers and snap the model back to whole integers.
• The Verdict: The cut proved that R$ 20,600.00 (Best objective) is the absolute cheapest integer combination possible. Because the best solution found matches the absolute proven limit (Best bound), the Gap hits 0.00%.
Summary: Your NVIDIA GPU mathematically proved and locked down the absolute most cost-effective fleet strategy in exactly 0.21 seconds.

In [10]:
status_atual = str(prob_maquinas.Status.name)

print("--- STATUS DO SOLVER ---")
print(f"Status da Solução: {status_atual}")

if "Optimal" in status_atual or "Feasible" in status_atual:
    print("\n--- ESTRATÉGIA DE CONTRATAÇÃO OTIMIZADA ---")
    print(f"Equipes Tipo A: {equipe_A.getValue()}")
    print(f"Equipes Tipo B: {equipe_B.getValue()}")
    
    # Formatação limpa do custo total acumulado
    custo_formatado = "{:,.2f}".format(prob_maquinas.ObjValue)
    print(f"Custo Mínimo: R$ {custo_formatado}")
else:
    print("\nO solucionador não encontrou uma solução viável com as restrições atuais.")


--- STATUS DO SOLVER ---
Status da Solução: Optimal

--- ESTRATÉGIA DE CONTRATAÇÃO OTIMIZADA ---
Equipes Tipo A: 2.0
Equipes Tipo B: 7.999999999999858
Custo Mínimo: R$ 20,600.00
